<a href="https://colab.research.google.com/github/2303A51553/HPC/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import time
from multiprocessing import Process, Queue
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt

# Create output folder
OUTPUT_DIR = "processed_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sample image paths
image_paths = ["image1.jpg", "image2.jpg", "image3.jpg", "image4.jpg"]

# Filters
FILTERS = {
    "Grayscale": lambda img: img.convert("L"),
    "Blur": lambda img: img.filter(ImageFilter.BLUR),
    "Sharpen": lambda img: img.filter(ImageFilter.SHARPEN),
    "Brightness": lambda img: ImageEnhance.Brightness(img).enhance(1.5)
}

# Sequential processing
def process_image(path):
    img = Image.open(path)
    results = {}
    for name, func in FILTERS.items():
        results[name] = func(img.copy())
    return results

start = time.time()
sequential_results = {}

for path in image_paths:
    sequential_results[path] = process_image(path)

serial_time = time.time() - start

# Parallel worker
def worker(paths, q):
    temp = {}
    for path in paths:
        temp[path] = process_image(path)
    q.put(temp)

# Parallel processing
start = time.time()
q = Queue()
processes = []

chunks = [image_paths[:2], image_paths[2:]]

for chunk in chunks:
    p = Process(target=worker, args=(chunk, q))
    p.start()
    processes.append(p)

parallel_results = {}

for _ in processes:
    parallel_results.update(q.get())

for p in processes:
    p.join()

parallel_time = time.time() - start
speedup = serial_time / parallel_time

# Save one sample output
sample = image_paths[0]
for name, img in parallel_results[sample].items():
    img.save(os.path.join(OUTPUT_DIR, f"{name}.png"))

# Graph: Serial vs Parallel Time
plt.bar(["Sequential", "Parallel"], [serial_time, parallel_time])
plt.title("Execution Time Comparison")
plt.ylabel("Time (seconds)")
plt.show()

# Graph: Speedup
plt.bar(["Speedup"], [speedup])
plt.title("Parallel Speedup")
plt.ylabel("Speedup Factor")
plt.show()

# Summary
print("Serial Time:", round(serial_time, 4), "seconds")
print("Parallel Time:", round(parallel_time, 4), "seconds")
print("Speedup:", round(speedup, 2), "x")

FileNotFoundError: [Errno 2] No such file or directory: 'image1.jpg'